<a href="https://colab.research.google.com/github/j-hay-214/osu-gradtda-5622-sp25/blob/main/Risk_States_Screener_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Establish Dependencies

In [ ]:
# Establishing dependencies

import os
import pandas as pd
import sqlite3 as sq

# Displays all rows of a df instead of truncating the output
# 'None' means there is no limit on number of rows shown
pd.set_option('display.max_rows', None)

# Displays all columns (useful for wide dataframes)
pd.set_option('display.max_columns', None)

# Ensures df output spans full display width instead of wrapping
# into multiple lines
pd.set_option('display.width', None)

# Prevents truncating long string values in cells
# 'None'
pd.set_option('display.max_colwidth', None)

In [ ]:
# Connect to a SQLite database (will be created if it doesn't exist).
def connect_db(file_name):
    if not os.path.isfile(file_name):
        print('Database file does not exist and will be created.')
    try:
        conn = sq.connect(file_name)
        print("Database connection established.")
        return conn
    except Error:
        print(Error)
        return None

"""
Make sure to close the connection before closing out of the file.
This will be the last function listed on the page.
Just run the code chunk and then close.
"""

In [ ]:
# Create a cursor to use to access the data.

"""
This is useful for allowing for the processing of individual rows.
Avoids only being able to handle the entire result set at once (SELECT)
Useful for loops or individual row edits
"""

def get_cursor(connection):
    try:
        cur = connection.cursor()
        print('Cursor created.')
        return cur
    except Error:
        print(Error)
        return None

In [ ]:
# Create a SQLite table by reading a worksheet in an excel file.  Uses Pandas.
# ASPIS data saved as .xlsx, so this is the function we would want to use.
def create_table_from_excel(connection, file_name, table_name):
    pdf = pd.read_excel(file_name)
    create_table_from_dataframe(connection, pdf, table_name)

In [ ]:
# Create a SQLite table from a Pandas DataFrame.
def create_table_from_dataframe(connection, pdf, table_name):
    pdf.to_sql(table_name, connection, if_exists='replace', index = False)

In [ ]:
# Create a Pandas DataFrame from a SQLite cursor.
def create_dataframe_from_cursor(cursor):
    cols = [column[0] for column in cursor.description]
    return pd.DataFrame.from_records(data = cursor.fetchall(), columns = cols)

In [ ]:
# Create a Pandas DataFrame from a SQLite table (or view).
#   Or could do: df = pd.read_sql_query("SELECT * FROM table_name", cnx)
#   See: https://stackoverflow.com/questions/36028759/how-to-open-and-convert-sqlite-database-to-pandas-dataframe
def create_dataframe_from_table(connection, table_name):
    cursor = run_query(connection,'SELECT * FROM "' + table_name + '"')
    cols = [column[0] for column in cursor.description]
    return pd.DataFrame.from_records(data = cursor.fetchall(), columns = cols)

In [ ]:
# Run a SQLite query and return the result.
def run_query(conn_or_cur, query_string):
    try:
        result = conn_or_cur.execute(query_string)
    except Exception as e:
        print("Query error:", e)
        result = None
    return result

In [ ]:
# Print query results up to a specified max number of records).
def print_result(cursor, max_num_records):
    print("------------------------------------------")
    df = create_dataframe_from_cursor(cursor)
    #print(df.head(max_num_records))
    display(df.head(max_num_records))
    print("------------------------------------------")

In [ ]:
# Print table or view for clean visual output
def print_table_or_view(conn_or_cur, table_name, max_num_records):
    print("\nTABLE/VIEW: " + table_name)
    result = run_query(conn_or_cur,'SELECT * FROM "' + table_name + '"')
    print_result(result, max_num_records)

In [ ]:
# Connect to the database and get a cursor to access the data.
#conn = connect(r'SQLiteUSStatesData.db')  # a file database
conn = connect_db(r':memory:')             # an in-memory database
cur = get_cursor(conn)  # not needed for these examples, but kept for compatibility

# Importing the Data

In [ ]:
from google.colab import files

# Prompt to upload file
data_upload = files.upload()

# Adjust file to be readable by pandas
# Uploaded files saved in dictionary format, so this requires adjustment
for file in data_upload.keys():
    data_df = pd.read_csv(file)

In [ ]:
# Creates a SQL table from the Pandas Dataframe
# Using function(cursor, df name, new table name)
create_table_from_dataframe(conn, data_df, 'data_table')

# Printing a small view to ensure that it works
print_table_or_view(conn, "data_table", 10)

# Select Columns of Interest

In [ ]:
# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_table')

# Creating new table with selected columns
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_table
    AS SELECT record_id AS id, screen_on_consent AS consented, screen_lat AS lat, screen_long AS long, age_43c95d AS age,
        recruitment_source AS source, screen_valid_check AS validity, interested AS interested, screen_eligibility AS eligibility,
        first_name, last_name, preferred_email AS email, secondary_email, phone_num AS phone,
        contact AS contacted
    FROM data_table
''')

# Converting to dataframe for accessing variable types in the next step
# Printing table to see if successful
rs_table_df = create_dataframe_from_table(conn, 'rs_table')
rs_table_df.head(15)

In [ ]:
"""
Most important were name, email, and phone being object as these are string values.
Additionally, lat and long should be float64 for comparing to eliminate those
outside of the U.S.
"""

# Inspecting data types to ensure proper formatting for manipulation
rs_table_df.dtypes

# Statuses

## Separating Already Contacted

In [ ]:
# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_contacted')

# Creating new table with selected columns
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_contacted
    AS SELECT *
    FROM rs_table
    WHERE rs_table.contacted == 1
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_contacted", 10)

In [ ]:
# Adding column with status of '1' to indicate that they were contacted already
# ONLY NEEDS TO BE RUN ONCE WHEN TABLE CREATED

# Adding new column of type integer to add status
run_query(conn, '''
    ALTER TABLE rs_contacted
    ADD COLUMN status INTEGER
    ''')

# Give status = 1 to all within this table as these are the ones who have already been contacted
run_query(conn, '''
    UPDATE rs_contacted
    SET status = 1
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_contacted", 10)

# Separating Not Contacted

In [ ]:
# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_other')

# Creating new table with selected columns
# Grabbing those where "contacted" = NaN as the only instances are NaN or 1.0
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_other
    AS SELECT *
    FROM rs_table
    WHERE rs_table.contacted IS NULL
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_other", 10)

In [ ]:
# Printing total number of rows to see how many we are starting with
row_count = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(row_count)

## Deemed Ineligible

### Latitude/Longitude Violations

In [ ]:
# ONLY NEEDS TO BE RUN ONCE

# Adding new column of type integer for status
run_query(conn, '''
    ALTER TABLE rs_other
    ADD COLUMN status INTEGER
    ''')

In [ ]:
"""
Only including mainland states to avoid complications.
Northernmost Latitude: 49.384358
Southern: 24.396308
Western: -125.0
Eastern: -66.93457
"""

# Update new column to be status of 2 if latitude and longitude are outside bounds
run_query(conn, '''
    UPDATE rs_other
    SET status = CASE
        WHEN lat IS NOT NULL AND long IS NOT NULL AND
        (lat < 24.396308 OR lat > 49.384358
            OR long < -125.0 OR long > -66.93457)
        THEN 2
        ELSE 0
    END
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_other", 10)

In [ ]:
# Separating those with lat/long violations

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_lat_long')

# Creating new table with selected columns
# Grabbing those where 'status' = 2
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_lat_long
    AS SELECT *
    FROM rs_other
    WHERE rs_other.status == 2
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_lat_long", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
lat_long_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_lat_long").fetchone()[0]
print(lat_long_row_count)

In [ ]:
# Removing the ones with lat/long violations from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_lat_long
        WHERE rs_other.id = rs_lat_long.id)
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_other", 10)

In [ ]:
# Printing total number of rows to see how many we have after removing lat/long
new_count_1 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {row_count}')
print(f'Number of Violations: {lat_long_row_count}')
print(f'New Number of Rows: {new_count_1}')

### Declined to Consent

In [ ]:
# Separating those that declined to consent

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_no_consent')

# Creating new table with selected columns where 'consented' is not '1'
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_no_consent
    AS SELECT *
    FROM rs_other
    WHERE rs_other.consented <> 1
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_no_consent", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
no_consent_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_no_consent").fetchone()[0]
print(no_consent_row_count)

In [ ]:
# Give status = 3 to all within this table as these are the ones who declined to consent
run_query(conn, '''
    UPDATE rs_no_consent
    SET status = 3
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_no_consent", 10)

In [ ]:
# Removing the ones with declined consent violations from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_no_consent
        WHERE rs_other.id = rs_no_consent.id)
''')

# Printing total number of rows to see how many we have after removing declining to consent
new_count_2 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_1}')
print(f'Number of Violations: {no_consent_row_count}')
print(f'New Number of Rows: {new_count_2}')

### Validity Error

In [ ]:
# Separating those that failed the validity check

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_validity')

# Creating new table with selected columns where 'validity' > 0
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_validity
    AS SELECT *
    FROM rs_other
    WHERE rs_other.validity > 0
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_validity", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
validity_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_validity").fetchone()[0]
print(validity_row_count)

In [ ]:
# Give status = 4 to all within this table as these failed validity check
run_query(conn, '''
    UPDATE rs_validity
    SET status = 4
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_validity", 10)

In [ ]:
# Removing the ones with validity violations from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_validity
        WHERE rs_other.id = rs_validity.id)
''')

# Printing total number of rows to see how many we have after removing validity errors
new_count_3 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_2}')
print(f'Number of Violations: {validity_row_count}')
print(f'New Number of Rows: {new_count_3}')

### Not Interested After Consenting

In [ ]:
# Separating those that indicated they were not interested

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_interest')

# Creating new table with selected columns where 'interested' <> 1
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_interest
    AS SELECT *
    FROM rs_other
    WHERE rs_other.interested <> 1.0
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_interest", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
interest_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_interest").fetchone()[0]
print(interest_row_count)

In [ ]:
# Give status = 5 to all within this table as these indicated no interest
run_query(conn, '''
    UPDATE rs_interest
    SET status = 5
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_interest", 10)

In [ ]:
# Removing the ones with no interest from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_interest
        WHERE rs_other.id = rs_interest.id)
''')

# Printing total number of rows to see how many we have after removing not interested
new_count_4 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_3}')
print(f'Number of Violations: {interest_row_count}')
print(f'New Number of Rows: {new_count_4}')

### Does Not Meet Criteria

In [ ]:
# Separating those that did not meet study criteria

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_criteria')

# Creating new table with selected columns where criteria is not met
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_criteria
    AS SELECT *
    FROM rs_other
    WHERE rs_other.eligibility <> "Eligible"
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_criteria", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
criteria_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_criteria").fetchone()[0]
print(criteria_row_count)

In [ ]:
# Give status = 6 to all within this table as these did not meet criteria
run_query(conn, '''
    UPDATE rs_criteria
    SET status = 6
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_criteria", 10)

In [ ]:
# Removing the ones that did not meet criteria from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_criteria
        WHERE rs_other.id = rs_criteria.id)
''')

# Printing total number of rows to see how many we have after removing those not meeting criteria
new_count_5 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_4}')
print(f'Number of Violations: {criteria_row_count}')
print(f'New Number of Rows: {new_count_5}')

### Invalid Recruitment Source

In [ ]:
# Separating those that provided an invalid recruitment source

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_recruit')

# Creating new table with selected columns where recruitment source was invalid
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_recruit
    AS SELECT *
    FROM rs_other
    WHERE rs_other.source == 3.0 OR rs_other.source == 5.0
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_recruit", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
recruit_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_recruit").fetchone()[0]
print(recruit_row_count)

In [ ]:
# Give status = 7 to all within this table as these provided invalid recruitment sources
run_query(conn, '''
    UPDATE rs_recruit
    SET status = 7
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_recruit", 10)

In [ ]:
# Removing the ones that provided invalid recruitment sources from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_recruit
        WHERE rs_other.id = rs_recruit.id)
''')

# Printing total number of rows to see how many we have after removing those with invalid recruitment sources
new_count_6 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_5}')
print(f'Number of Violations: {recruit_row_count}')
print(f'New Number of Rows: {new_count_6}')

### Null After Consent

In [ ]:
# Separating those that provided no information after consent

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_null')

# Creating new table with selected columns where nothing is provided
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_null
    AS SELECT *
    FROM rs_other
    WHERE rs_other.eligibility IS NULL
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_null", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
null_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_null").fetchone()[0]
print(null_row_count)

In [ ]:
# Give status = 8 to all within this table as they provided nothing after consent
run_query(conn, '''
    UPDATE rs_null
    SET status = 8
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_null", 10)

In [ ]:
# Removing the ones that did not meet criteria from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_null
        WHERE rs_other.id = rs_null.id)
''')

# Printing total number of rows to see how many we have after removing those with null responses
new_count_7 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_6}')
print(f'Number of Violations: {null_row_count}')
print(f'New Number of Rows: {new_count_7}')

### Did Not Provide Contact Information

In [ ]:
# Separating those that did not provide contact information

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_contact')

# Creating new table with selected columns where contact is not provided
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_contact
    AS SELECT *
    FROM rs_other
    WHERE rs_other.email IS NULL AND rs_other.secondary_email IS NULL AND rs_other.phone IS NULL
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_contact", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
contact_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_contact").fetchone()[0]
print(contact_row_count)

In [ ]:
# Give status = 9 to all within this table as these did not provide contact information
run_query(conn, '''
    UPDATE rs_contact
    SET status = 9
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_contact", 10)

In [ ]:
# Removing the ones that did not provide contact information from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_contact
        WHERE rs_other.id = rs_contact.id)
''')

# Printing total number of rows to see how many we have after removing those without contact information
new_count_8 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_7}')
print(f'Number of Violations: {contact_row_count}')
print(f'New Number of Rows: {new_count_8}')

### Duplicate Entries (Eligible)

In [ ]:
# Separating those that provided duplicate entries

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_duplicates')

# Creating new table with selected columns where there are duplicates in the contact information
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_duplicates
    AS SELECT *, rs_other.email
    FROM rs_other
    INNER JOIN (
        SELECT email
        FROM rs_other
        GROUP BY email
        HAVING COUNT(*) > 1
) rs_duplicates ON rs_other.email = rs_duplicates.email
''')

# Printing table to see if successful
print_table_or_view(conn, "rs_duplicates", 10)

In [ ]:
# Printing total number of rows to see how many fall into this status currently
duplicates_row_count = run_query(conn, "SELECT COUNT(*) FROM rs_duplicates").fetchone()[0]
print(duplicates_row_count)

In [ ]:
# Only needs to be run once!

# Dropping the extra email columns added to the table
run_query(conn, '''
    ALTER TABLE rs_duplicates
    DROP COLUMN `email:1`
    ''')

run_query(conn, '''
    ALTER TABLE rs_duplicates
    DROP COLUMN `email:2`
    ''')

In [ ]:
# Give status = 10 to all within this table as these did not provide contact information
run_query(conn, '''
    UPDATE rs_duplicates
    SET status = 10
    ''')

# Printing table to see if successful
print_table_or_view(conn, "rs_duplicates", 10)

In [ ]:
# Removing the ones that provided duplicates from the main table
run_query(conn, '''
    DELETE FROM rs_other
    WHERE EXISTS (
        SELECT 1
        FROM rs_duplicates
        WHERE rs_other.id = rs_duplicates.id)
''')

# Printing total number of rows to see how many we have after removing those without contact information
new_count_9 = run_query(conn, "SELECT COUNT(*) FROM rs_other").fetchone()[0]
print(f'Original Number of Rows: {new_count_8}')
print(f'Number of Violations: {duplicates_row_count}')
print(f'New Number of Rows: {new_count_9}')

# Displaying Remaining Records

In [ ]:
# Showing first 15 rows to see if things look correct
print_table_or_view(conn, "rs_other", 15)

## Standardizing Names

In [ ]:
# Standardizing first and last name to have the first letter capitalized and the rest lower case
run_query(conn, '''
    UPDATE rs_other
    SET first_name = UPPER(SUBSTR(first_name, 1, 1)) || LOWER(SUBSTR(first_name, 2))
    ''')

run_query(conn, '''
    UPDATE rs_other
    SET last_name = UPPER(SUBSTR(last_name, 1, 1)) || LOWER(SUBSTR(last_name, 2))
    ''')

# Showing first 15 rows to see if things look correct
print_table_or_view(conn, "rs_other", 15)

## Standardizing Emails

In [ ]:
# Standardizing email to be all lower case
run_query(conn, '''
    UPDATE rs_other
    SET email = LOWER(email)
    ''')

# Showing first 15 rows to see if things look correct
print_table_or_view(conn, "rs_other", 15)

# Combining Ineligible Participants

In [ ]:
# Combining the tables of all ineligible reasons

# Delete the previous table if it already exists
run_query(conn,'DROP TABLE IF EXISTS rs_ineligible_total')

# Creating new table with combination of all ineligible groups with accurate statuses
run_query(conn,'''
    CREATE TABLE IF NOT EXISTS rs_ineligible_total
    AS SELECT *
    FROM rs_lat_long
    UNION ALL
    SELECT *
    FROM rs_no_consent
    UNION ALL
    SELECT *
    FROM rs_validity
    UNION ALL
    SELECT *
    FROM rs_interest
    UNION ALL
    SELECT *
    FROM rs_criteria
    UNION ALL
    SELECT *
    FROM rs_recruit
    UNION ALL
    SELECT *
    FROM rs_null
    UNION ALL
    SELECT *
    FROM rs_contact
    UNION ALL
    SELECT *
    FROM rs_duplicates
    ORDER BY id
    ''')

# Showing first 15 rows to see if things look correct
print_table_or_view(conn, "rs_ineligible_total", 15)

## Updating Recruitment Source for Output

In [ ]:
# Updating recruitment source for output
run_query(conn, '''
    UPDATE rs_ineligible_total
    SET source = CASE
        WHEN CAST(source AS TEXT) = '1.0' THEN 'American Population Panel'
        WHEN CAST(source AS TEXT) = '2.0' THEN 'ResearchMatch'
        WHEN CAST(source AS TEXT) = '3.0' THEN 'Social Media Advertisement'
        WHEN CAST(source AS TEXT) = '4.0' THEN 'STRIVE Registry'
        WHEN CAST(source as TEXT) = '5.0' THEN 'StudySearch'
        WHEN CAST(source as TEXT) = '6.0' THEN 'Other'
        ELSE source
    END
    ''')

# Showing first 15 rows to see if things look correct
print_table_or_view(conn, "rs_ineligible_total", 15)

## Updating Status for Output

In [ ]:
# Updating status for output
run_query(conn, '''
    UPDATE rs_ineligible_total
    SET status = CASE
        WHEN CAST(status AS TEXT) = '1' THEN 'Already contacted'
        WHEN CAST(status AS TEXT) = '2' THEN 'Latitude/Longitude violation'
        WHEN CAST(status AS TEXT) = '3' THEN 'Declined to consent'
        WHEN CAST(status AS TEXT) = '4' THEN 'Validity error after consenting'
        WHEN CAST(status as TEXT) = '5' THEN 'Not interested after consenting'
        WHEN CAST(status as TEXT) = '6' THEN 'Did not meet study criteria after consenting'
        WHEN CAST(status as TEXT) = '7' THEN 'Invalid recruitment source provided'
        WHEN CAST(status as TEXT) = '8' THEN 'Null response after consenting'
        WHEN CAST(status as TEXT) = '9' THEN 'Did not provide contact information after consenting'
        WHEN CAST(status as TEXT) = '10' THEN 'Duplicate entry'
        ELSE status
    END
    ''')

# Showing first 15 rows to see if things look correct
print_table_or_view(conn, "rs_ineligible_total", 15)

# Write Ineligible Participants to Template

In [ ]:
# Converting to df for output to Excel
rs_ineligible_df = create_dataframe_from_table(conn, 'rs_ineligible_total')

In [ ]:
from openpyxl import load_workbook

# Defining function to write the ineligible data to the excel template
def write_ineligible_to_template(df, template_path, output_path):

    # Reading template to write on
    workbook = load_workbook(template_path)     # Load template for writing on
    sheet = workbook["Sheet1"]  # Selecting the first sheet

    # Establish starting row
    starting_row = 2        # To account for the headers in the template

    # Write id values onto the document
    for i, value in enumerate(df["id"], start=starting_row):
        sheet[f'A{i}'] = value

    # Write Recruitment source onto the document
    for i, value in enumerate(df["source"], start=starting_row):
        sheet[f'B{i}'] = value

    # Write status onto the document
    for i, value in enumerate(df["status"], start=starting_row):
        sheet[f'C{i}'] = value

    # Save the workbook for output path
    workbook.save(output_path)

    # Download directly to computer
    files.download(output_path)

    # Print confirmation message
    print(f"Saved data for {output_path}")

In [ ]:
from datetime import datetime

# Upload the template for ineligible participants output
ineligible_template = files.upload()

# Load the excel template
template_path = list(ineligible_template.keys())[0]

# Identify current date for output
current_date = datetime.now().strftime("%m-%d-%Y")

# Identify output path
output_path = (f"FOCUS_Ineligible_{current_date}.xlsx")

# Run the function on the ineligible participants table
write_ineligible_to_template(rs_ineligible_df, template_path, output_path)

# Write Eligible Participants to Template

In [ ]:
# Converting to df for output to Excel
rs_eligible_df = create_dataframe_from_table(conn, 'rs_other')

In [ ]:
from openpyxl import load_workbook

# Defining function to write the eligible data to the excel template
def write_eligible_to_template(df, template_path, output_path):

    # Reading template to write on
    workbook = load_workbook(template_path)     # Load template for writing on
    sheet = workbook["Sheet1"]  # Selecting the first sheet

    # Establish starting row
    starting_row = 2        # To account for the headers in the template

    # Write id values onto the document
    for i, value in enumerate(df["id"], start=starting_row):
        sheet[f'A{i}'] = value

    # Write first name onto the document
    for i, value in enumerate(df["first_name"], start=starting_row):
        sheet[f'B{i}'] = value

    # Write last name onto the document
    for i, value in enumerate(df["last_name"], start=starting_row):
        sheet[f'C{i}'] = value

    # Write primary email onto the document
    for i, value in enumerate(df["email"], start=starting_row):
        sheet[f'D{i}'] = value

    # Write secondary email onto the document
    for i, value in enumerate(df["secondary_email"], start=starting_row):
        sheet[f'E{i}'] = value

    # Write phone number onto the document
    for i, value in enumerate(df["phone"], start=starting_row):
        sheet[f'F{i}'] = value

    # Save the workbook for output path
    workbook.save(output_path)

    # Download directly to computer
    files.download(output_path)

    # Print confirmation message
    print(f"Saved data for {output_path}")

In [ ]:
from datetime import datetime

# Upload the template for eligible participants output
eligible_template = files.upload()

# Load the excel template
eligible_template_path = list(eligible_template.keys())[0]

# Identify current date for output
current_date = datetime.now().strftime("%m-%d-%Y")

# Identify output path
eligible_output_path = (f"FOCUS_Eligible_{current_date}.xlsx")

# Run the function on the ineligible participants table
write_eligible_to_template(rs_eligible_df, eligible_template_path, eligible_output_path)

# Write Eligible Participants to Template

# Closing Connection

In [ ]:
# Close the connection to the database.
def close_db(connection, cursor):
    connection.commit()
    cursor.close()
    connection.close()

In [ ]:
close_db(conn, cur)